# Hybrid Transformer vs. Standard Transformer Comparison

This notebook compares the standard Probabilistic Transformer against the Hybrid Transformer which integrates an Ornstein-Uhlenbeck (OU) process to model residuals

## Methodology

1.  Standard transformer: predicting prices solely based on input features
2.  Hybrid Transformer:
    - Train Transformer for trend prediction
    - Model residuals (Actual - Predicted) using an OU process
    - Final forecast = transformer trend + OU process mean reversion

We run both models 10 times to reduce statistical variability and report average metrics (MAE, RMSE, MAPE, R2, Pinball Loss)

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from models import ProbabilisticTransformer, HybridProbabilisticTransformer
from core.experiment_utils import (
    load_data, load_cache, save_cache, run_experiment, N_RUNS,
)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

2026-03-08 12:45:27.617452: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772970327.723877  962680 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772970327.759102  962680 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772970328.011390  962680 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772970328.011432  962680 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772970328.011438  962680 computation_placer.cc:177] computation placer alr

GPUs detected: 1


In [2]:
RESULTS_DIR = Path(project_root) / "results" / "hybrid_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "results.json"

In [3]:
# All metrics and caching handled by core.experiment_utils

In [4]:
data = load_data()


Data loaded  —  Train: 10367, Val: 1997, Test: 1128


In [5]:
cache = load_cache(CACHE_FILE)

run_experiment(
    ProbabilisticTransformer, "Standard Transformer", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
)

run_experiment(
    HybridProbabilisticTransformer, "Hybrid (Transformer+OU)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=True,
)

save_cache(CACHE_FILE, cache)

[Standard Transformer] found on disk (/home/d1ff1cult/masterproef_new/results/hybrid_comparison/Standard_Transformer/summary.json) — skipping.

  Experiment: Hybrid (Transformer+OU)
  Model: HybridProbabilisticTransformer  |  head: johnson_su  |  runs: 10
  Run 1/10 ...


I0000 00:00:1772970334.520886  962680 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7483 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:01:00.0, compute capability: 6.1


Epoch 1/30


I0000 00:00:1772970340.946931  962901 service.cc:152] XLA service 0x7bf388004840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772970340.946968  962901 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1080, Compute Capability 6.1
2026-03-08 12:45:41.142903: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1772970342.167666  962901 cuda_dnn.cc:529] Loaded cuDNN version 90300


  6/324 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 1.6143

I0000 00:00:1772970350.058652  962901 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


324/324 ━━━━━━━━━━━━━━━━━━━━ 36s 66ms/step - loss: 1.0985 - val_loss: 1.0485 - learning_rate: 7.0000e-04
Epoch 2/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.7411 - val_loss: 0.9997 - learning_rate: 7.0000e-04
Epoch 3/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.5914 - val_loss: 0.9395 - learning_rate: 7.0000e-04
Epoch 4/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.4922 - val_loss: 0.9157 - learning_rate: 7.0000e-04
Epoch 5/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.4034 - val_loss: 1.1125 - learning_rate: 7.0000e-04
Epoch 6/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.3607 - val_loss: 0.9550 - learning_rate: 7.0000e-04
Epoch 7/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.3037 - val_loss: 1.1066 - learning_rate: 7.0000e-04
Epoch 8/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.2725 - val_loss: 1.1868 - learning_rate: 7.0000e-04
Epoch 9/30
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.2318 - val_loss: 1.

In [6]:
results = [{"Model": k, **v} for k, v in cache.items()]
df_res = pd.DataFrame(results)
if not df_res.empty:
    display(df_res.sort_values("MAE"))
    df_res.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

,Model,MAE,MSE,RMSE,MAPE,R2,PICP,MPIW,PINAW,IntervalScore,CRPS,NLL,Pinball_10,Pinball_50,Pinball_90,Avg_Pinball,training_time
0,Standard Transformer,21.151128,807.129051,28.362899,5928.792518,0.553383,0.928406,109.683585,0.342291,154.956502,15.614259,44.772191,5.671147,10.602566,4.623758,6.965823,97.530183
1,Hybrid (Transformer+OU),21.795338,850.041760,29.146209,5754.943061,0.529637,0.919596,109.586111,0.341986,163.950571,16.124928,NaN,5.607327,11.045499,5.113624,7.255483,104.061343
